# TACO + Gazebo + real webcam fine-tuning
Use a GPU runtime. Upload the listed files to `MyDrive/robot-1-colab/`.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/robot-1-colab'


In [ ]:
!pip -q install 'ultralytics>=8.4,<9'
!unzip -q -o $DRIVE/taco.zip -d /content/taco
!unzip -q -o $DRIVE/gazebo.zip -d /content/gazebo
!unzip -q -o $DRIVE/real_webcam.zip -d /content/real_webcam
!cp $DRIVE/mixed_real_data.yaml /content/mixed_real_data.yaml


In [ ]:
from ultralytics import YOLO
RUN_NAME = 'taco_gazebo_real_hardneg_20261028'
model = YOLO(f'{DRIVE}/taco_best.pt')
model.train(data='/content/mixed_real_data.yaml', imgsz=960, batch=4, epochs=50, patience=10, seed=42, deterministic=True, device=0, project=f'{DRIVE}/real_runs', name=RUN_NAME, save_period=5)
BEST_PATH = str(model.trainer.best)
print('trained best:', BEST_PATH)


In [ ]:
print('evaluating:', BEST_PATH)
best = YOLO(BEST_PATH)
mixed_metrics = best.val(data='/content/mixed_real_data.yaml', split='test', imgsz=960, device=0)
real_metrics = best.val(data='/content/real_webcam/data.yaml', split='test', imgsz=960, device=0, project=f'{DRIVE}/real_runs', name=f'{RUN_NAME}_real_test')
print('mixed mAP50:', mixed_metrics.box.map50)
print('real mAP50:', real_metrics.box.map50)
